# W02-A · From text to terms *(guided lab, ~40 min)*

**Business question:** A customer asks: *"Do these water filters leak?"* You have 30,000 reviews. Before you can
**search** them, you have to decide what counts as a word, and which words count as the same.

By the end of this notebook you will be able to:
1. explain why searching for a **string** is not the same as searching for a **word**
2. **tokenise** text three ways and count **tokens** and **types**
3. describe the shape of a vocabulary with **Zipf's law**
4. remove **stop words**, and say when that is a mistake
5. **stem** and **lemmatise** words, and say what each gains and loses for search

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shripada/ads6703-nlp/blob/main/w02/W02-A-text-to-terms.ipynb) · On Colab, run the next cell first. On your laptop, you can skip it.

In [ ]:
# Only for Google Colab: fetch the course folder (with the data) and move into this week's folder.
# On your laptop this cell does nothing.
import os, subprocess, sys
if "google.colab" in sys.modules:
    if os.path.exists("/content/ADS6703"):
        subprocess.run(["git", "-C", "/content/ADS6703", "pull", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", "https://github.com/shripada/ads6703-nlp.git", "/content/ADS6703"], check=True)
    os.chdir("/content/ADS6703/w02")

In [ ]:
# ▶ Setup: just run this cell. It loads the libraries, NLTK's word lists and the 30,000 reviews.
import re                          # regular expressions, from Workshop 1
from collections import Counter    # counting words (see W02-0)
import pandas as pd                # the review table
import matplotlib.pyplot as plt    # the Zipf chart in section 3
import nltk                        # the Natural Language Toolkit: tokeniser, stop words, stemmer, lemmatiser

# NLTK keeps its word lists and models in separate downloads. This fetches the ones we use, once per laptop;
# quiet=True hides the progress messages. If you're offline and already ran it once, it does nothing.
for pkg in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4"]:
    nltk.download(pkg, quiet=True)

pd.set_option("display.max_colwidth", 160)   # show 160 characters per cell, so review text isn't cut off

DATA = "../data/"   # notebooks live in ADS6703/w02/, data in ADS6703/data/
reviews = pd.read_csv(DATA + "reviews.csv.gz")
# As in W1: replace the HTML line breaks <br />, <br/> and <br> with a space
reviews["clean"] = reviews["text"].fillna("").str.replace(r"<br\s*/?>", " ", regex=True)
print(reviews.shape)   # expect (30000, 10): the 9 original columns plus "clean"

## 1 · Different approaches of search (5 min)

The quickest way to answer the customer is to search the text for `leak`, the way you would with Find in Excel or a browser.

In [ ]:
hits = reviews["clean"].str.contains("leak", case=False)   # True if "leak" appears anywhere in the text, any case
print(hits.sum(), "reviews contain the string 'leak'")

# Which actual words did that catch? Collect every word around "leak", from every review.
found = []
for text in reviews["clean"]:
    found += re.findall(r"\w*leak\w*", text.lower())   # \w*leak\w* grows each match to the whole word
for word, n in Counter(found).most_common():           # most frequent first
    print(f"{n:>5}  {word}")

> 📖 **Reading the output.** The string search finds 924 reviews. The list shows every *word* it matched, with how
> often it appears. Most are forms of the same word: *leaks, leak, leaking, leaked*. A few are typos where two words
> ran together (*orleaks*, *groundsleaked*). And *leakproof* is the opposite of a complaint.

Now try the opposite mistake: match only the **exact word** `leak`.

In [ ]:
exact = reviews["clean"].str.contains(r"\bleak\b")   # the whole word "leak", lower case only
print(exact.sum(), "reviews contain the word 'leak'")

Far fewer. The exact-word search misses *Leaks*, *leaking* and *leaked*, which are complaints about the same thing.

> 💬 **Discuss:** You want **every** review about leaking. Which is worse here, the string search or the exact-word search?
> Name one false positive of the first and one false negative of the second.

A search engine solves this in two steps, and they are the rest of this notebook:
1. **Tokenise:** cut the text into words (*tokens*), so `orleaks` and `cloudy` are not matched by accident.
2. **Normalise:** map the different forms of a word to one **term**, so *Leaks*, *leaking* and *leaked* all count as `leak`.

## 2 · Tokenisation (10 min)

A **token** is one occurrence of a word (or number, or symbol) in the text. A **type** is a distinct token.
*"the filter and the jug"* has 5 tokens and 4 types, because *the* appears twice. The set of types is the **vocabulary**.

Splitting on spaces is the obvious first attempt. Compare it with a regex and with NLTK's tokeniser on one awkward sentence.

In [ ]:
s = "Don't buy it! The filter (model UKF8001) cost $24.99 at Lowe's, and it leaked after 2-3 weeks..."

print(s.split())                # 1: split on whitespace
print(re.findall(r"\w+", s))    # 2: every run of word characters [A-Za-z0-9_]
print(nltk.word_tokenize(s))    # 3: NLTK's tokeniser, rules written for English news text

> 📖 **Reading the output.** Look at what each one does to the awkward bits:
>
> | | `split()` | `\w+` | NLTK |
> |---|---|---|---|
> | `Don't` | `Don't` | `Don`, `t` | `Do`, `n't` |
> | `it!` | `it!` (punctuation stuck on) | `it` | `it`, `!` |
> | `$24.99` | `$24.99` | `24`, `99` | `$`, `24.99` |
> | `Lowe's,` | `Lowe's,` | `Lowe`, `s` | `Lowe`, `'s`, `,` |
> | `2-3` | `2-3` | `2`, `3` | `2-3` |
>
> None of them is "right". `split()` leaves punctuation on words, so `it!` and `it` become two different types.
> `\w+` breaks contractions. NLTK keeps prices and ranges whole and splits *n't* off as its own token ("not"), which is
> what a grammar parser wants. **A tokeniser is a design choice**, and the right one depends on what you do next.

For search we want words and numbers, lower-cased, with no punctuation, and contractions kept whole.

### ✍️ Exercise 1: a tokeniser for search

Write `tokenize(text)`: **lower-case** the text, then return every match of this pattern, in order:

`[a-z0-9]+(?:'[a-z]+)?`: a run of letters or digits, optionally followed by an apostrophe and more letters (`don't`, `lowe's`).

In [ ]:
TOKEN = r"[a-z0-9]+(?:'[a-z]+)?"   # letters/digits, plus an optional 's or 't ending
def tokenize(text):
    return ...   # ✍️ lower-case the text, then re.findall(TOKEN, ...)
print(tokenize(s))
assert tokenize(s) == ["don't", "buy", "it", "the", "filter", "model", "ukf8001", "cost", "24", "99", "at",
                       "lowe's", "and", "it", "leaked", "after", "2", "3", "weeks"], "not quite: did you lower-case first?"
print("✅ Note what it still gets wrong: $24.99 became two tokens. Fine for search, bad for prices.")

Now tokenise all 30,000 reviews and compare the size of the vocabulary under each approach.

In [ ]:
reviews["tokens"] = reviews["clean"].apply(tokenize)   # one list of tokens per review

def count_words(token_lists):
    # token_lists: one list of tokens per review. Count every token in all of them.
    counts = Counter()
    for tokens in token_lists:
        for t in tokens:
            counts[t] += 1
    return counts

def report(name, token_lists):
    counts = count_words(token_lists)
    print(f"{name:<16} {sum(counts.values()):>9,} tokens  {len(counts):>6,} types")

report("split()", [text.split() for text in reviews["clean"]])
report("lower + split()", [text.lower().split() for text in reviews["clean"]])
report("tokenize()", reviews["tokens"])

> 📖 **Reading the output.** There are just over 1.1 million tokens whichever way you cut them. The number of
> **types** is what changes: 45,310 with `split()`, 39,488 after lower-casing, and 17,602 with our tokeniser, which
> strips the punctuation. Lower-casing alone merged about 6,000 types (`Filter` and `filter`), and dropping punctuation
> merged about 22,000 more (`filter`, `filter.`, `filter,`, `(filter`, …). A smaller vocabulary means a smaller index,
> and more matches for each search word.

## 3 · Zipf's law: a few words are everywhere, most are rare (5 min)

In [ ]:
freq = count_words(reviews["tokens"])   # how often each type appears, a Counter
n_tokens = sum(freq.values())

print(freq.most_common(15))                    # the 15 most frequent types

top100 = 0                                     # tokens covered by the 100 most frequent types
for word, n in freq.most_common(100):
    top100 += n
print(f"The top 100 types are {top100 / n_tokens:.0%} of all tokens")

once = len([w for w in freq if freq[w] == 1])  # types seen exactly once
print(f"{once:,} of {len(freq):,} types ({once / len(freq):.0%}) appear exactly once")

In [ ]:
# ▶ Just run this cell and read the output: you don't need to follow the code.
# Rank-frequency chart: the most frequent type is rank 1, the next rank 2, ...
counts = sorted(freq.values(), reverse=True)
ranks = range(1, len(counts) + 1)
plt.figure(figsize=(6, 4))
plt.loglog(ranks, counts)                # log scale on both axes
plt.xlabel("rank (log scale)")
plt.ylabel("frequency (log scale)")
plt.title("Word frequency against rank in 30,000 reviews")
plt.show()

> 📖 **Reading the output.** The 15 most frequent types are nearly all small "function" words (*the, i, and, to*).
> *coffee* is the first content word. Just 100 types make up over half of all the text, while about 42% of the
> vocabulary appears only once (typos, rare model numbers, unusual words). On log–log axes the curve is close to a
> straight line: this is **Zipf's law**, frequency ∝ 1 / rank. It holds in every language and every collection.
>
> For search it means two things. The most frequent words appear in almost every review, so they are useless for
> telling reviews apart. And most of the vocabulary is rare, so the index has a long tail of short entries.

## 4 · Stop words (5 min)

**Stop words** are very frequent words that carry little meaning on their own, like *the, a, of, is*. Many systems
drop them before indexing. NLTK has a ready-made list for English.

In [ ]:
from nltk.corpus import stopwords
STOP = set(stopwords.words("english"))   # a set, for fast `in` checks
print(len(STOP), "stop words, e.g.", sorted(STOP)[:12])

stop_tokens = sum([freq[w] for w in STOP])     # how often stop words occur (a Counter gives 0 for a missing word)
stop_types = len([w for w in freq if w in STOP])
print(f"Removing them drops {stop_tokens / n_tokens:.0%} of all tokens, "
      f"but only {stop_types} of {len(freq):,} types")

Half the text disappears, which makes an index much smaller. But look at what is on the list:

In [ ]:
print(sorted([w for w in STOP if w in {"no", "not", "nor"} or w.endswith("n't")]))
print([w for w in tokenize("No leaks, and it doesn't smell") if w not in STOP])   # what survives

> 📖 **Reading the output.** *no, not* and every *n't* contraction are stop words. After removal,
> *"No leaks, and it doesn't smell"* becomes `['leaks', 'smell']`: the meaning has flipped from praise to complaint.
> Search engines today mostly **keep** stop words (disk is cheap, and phrases like *"to be or not to be"* or the band *The Who*
> are made of them). Sentiment analysis in Workshop 4 needs the negations too.

### ✍️ Exercise 2: stop words, but keep the negations

Make `MY_STOP`: NLTK's list **minus** `no`, `not`, `nor` and every word ending in `n't`. Then write `remove_stop(tokens)`.

In [ ]:
NEGATIONS = {w for w in STOP if ...}   # ✍️ no, not, nor, and words ending in "n't" (hint: w.endswith)
MY_STOP = ...                          # ✍️ STOP without the negations

def remove_stop(tokens):
    return ...   # ✍️ a list comprehension
print(remove_stop(tokenize("No leaks, and it doesn't smell")))
assert remove_stop(tokenize("No leaks, and it doesn't smell")) == ["no", "leaks", "doesn't", "smell"], "not quite"
assert "the" in MY_STOP and "not" not in MY_STOP, "MY_STOP should still contain 'the', but not 'not'"
print("✅")

## 5 · Stemming and lemmatisation (10 min)

Tokenising and lower-casing still leave *leak, leaks, leaking* and *leaked* as four different types. Two ways to merge them:

- A **stemmer** chops off endings by rule, without a dictionary. The **Porter stemmer** (1980) applies about 60 rules
  in steps: *sses → ss*, *ies → i*, drop *s*, drop *ed* or *ing* if a vowel is left, and so on. The result, the **stem**,
  need not be a real word.
- A **lemmatiser** looks the word up in a dictionary (here WordNet) and returns its **lemma**, the form you'd find in
  the dictionary: *ran → run*, *mice → mouse*. It needs to know the **part of speech**: is *leaking* a verb or a noun?

In [ ]:
from nltk.stem import PorterStemmer, WordNetLemmatizer
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

words = ["leaks", "leaking", "leaked", "leakage", "leaky",
         "replacement", "replaced", "replacing",
         "general", "generic", "generous",
         "filters", "fitness", "ran", "was", "ponies", "better"]
pd.DataFrame({
    "Porter stem": [stemmer.stem(w) for w in words],
    "lemma (as noun)": [lemmatizer.lemmatize(w) for w in words],          # the default part of speech is noun
    "lemma (as verb)": [lemmatizer.lemmatize(w, pos="v") for w in words], # pos="v": treat it as a verb
}, index=words)

> 📖 **Reading the output.** Read it row by row:
>
> - **Stemming merges forms well:** *leaks, leaking, leaked → leak* and *replacement, replaced, replacing → replac*. The stem *replac* isn't a word, but nobody sees it; it only has to be the same for all three.
> - **Over-stemming** (too much merged): *general, generic, generous → gener*. A search for "generic filter" will also return "generally works fine". This hurts **precision**.
> - **Under-stemming** (too little merged): *leakage → leakag* and *leaky → leaki* stay apart from *leak*, and *ran* stays *ran*. This hurts **recall**.
> - **The lemmatiser** only returns real words, and it knows *ran → run* and *was → be*, but only when told they are verbs. As nouns it leaves *leaking* alone, and turns *was* into *wa* (it reads it as the plural of "wa"). Getting the part of speech right needs a tagger, which is slower and makes its own mistakes.

How much does each one shrink the vocabulary?

In [ ]:
vocab = list(freq)   # the 17,602 types
stems = {w: stemmer.stem(w) for w in vocab}           # each type -> its stem (a dict, so we can reuse it below)
lemmas = {w: lemmatizer.lemmatize(w) for w in vocab}  # each type -> its noun lemma
# set() drops the repeats, so len(set(...)) is the number of distinct stems (or lemmas)
print(f"types: {len(vocab):,}   after stemming: {len(set(stems.values())):,}   "
      f"after lemmatising: {len(set(lemmas.values())):,}")

### ✍️ Exercise 3: which words share a stem?

The words that a stemmer merges into one stem are called a **conflation class**. Build `by_stem`, a dict from each
stem to the **list of types** that have it. (This is the dict-of-lists pattern from W02-0, section 5.)

In [ ]:
by_stem = {}
for w, st in stems.items():
    ...   # ✍️ add w to the list stored under st (hint: setdefault)
print(sorted(by_stem["leak"]))
print(sorted(by_stem["gener"]))
assert sorted(by_stem["leak"]) == ["leak", "leaked", "leaking", "leaks"], "not quite: check that every type is added"
print("✅")

In [ ]:
# The six biggest classes: sort the stems by the length of their list, longest first (W02-0, section 6)
biggest = sorted(by_stem, key=lambda st: len(by_stem[st]), reverse=True)[:6]
for st in biggest:
    print(f"{st:>8}: {sorted(by_stem[st])}")

> 📖 **Reading the output.** Most big classes are good merges: every form of *install*, *remove* or *replace*. But
> look at *gener*, and at misspellings like *replacment* and *noticible*, which happen to land in the right class.

## 6 · What this does to a search (5 min)

Back to the customer's question. Count the reviews that contain **the token `leak`**, then the reviews that contain
**any token whose stem is `leak`**.

In [ ]:
def stem_all(tokens):
    return [stems[t] for t in tokens]   # each token replaced by its stem, from the dict above

reviews["stems"] = reviews["tokens"].apply(stem_all)   # one list of stems per review

def search(column, word):
    # For every review: True if `word` is in its list in `column` ("tokens" or "stems"), otherwise False
    found = [word in items for items in reviews[column]]
    return pd.Series(found)   # a True/False column, like the one str.contains gives

has_token = search("tokens", "leak")   # exact token
has_stem = search("stems", "leak")     # any token whose stem is "leak"
print(f"token 'leak': {has_token.sum()} reviews   stem 'leak': {has_stem.sum()} reviews")

has_generic = search("tokens", "generic")
has_gener = search("stems", "gener")
print(f"token 'generic': {has_generic.sum()} reviews   stem 'gener': {has_gener.sum()} reviews")

In [ ]:
# ▶ Just run this cell and read the output: you don't need to follow the code.
# A few of the extra reviews the stem brings in for "generic": do they mention generic products?
extra = reviews.loc[has_gener & ~has_generic, "clean"]   # matched by the stem "gener", but no token "generic"
for text in extra.head(4):
    m = re.search(r"(?i)\bgener\w*", text)               # find the gener- word and print 60 characters around it
    print("…" + text[max(0, m.start() - 60): m.end() + 40] + "…")

> 📖 **Reading the output.** For *leak*, stemming nearly **triples** the number of reviews found, from 320 to 885,
> and every extra one really uses a form of the word: **recall** goes up. For *generic*, the stem adds about 100 reviews
> about things being *generally* fine or *generous* portions: **precision** goes down.
>
> **Manager's lens.** The same trade-off as Workshop 1, one level up. Normalising more (case folding, stemming)
> finds more of what you want and more of what you don't. A support team searching for *every* leak complaint
> wants the recall. A shopper typing "generic filter" into a store's search box wants the precision.

Finally, read some of the 885 *leak* reviews:

In [ ]:
# ▶ Just run this cell and read the output: you don't need to follow the code.
# The pattern: a negation (no, not, never, without, zero or any n't word), at most one more word, then a leak-word.
# It matches "no leaks", "doesn't leak", "won't ever leak", ...
leak_reviews = reviews[has_stem]   # the 885 reviews with a token whose stem is "leak"
negated = leak_reviews["clean"].str.contains(r"(?i)\b(?:no|not|never|without|zero|\w+n't)\s+(?:\w+\s+)?leak")
print(f"{negated.sum()} of {len(leak_reviews)} leak reviews have a negation just before the leak word")
print(f"average rating: with the negation {leak_reviews[negated]['rating'].mean():.2f}, "
      f"without {leak_reviews[~negated]['rating'].mean():.2f}")
leak_reviews[negated][["rating", "clean"]].head(4)

> 📖 **Reading the output.** Nearly half of the "leak" reviews put a negation right before the leak word, and they
> average about 4.6★ against about 3.6★ for the rest. Most are saying the product **doesn't** leak (*"so it won't leak"*).
> A few are still complaints (review 122: *"Leaked after 5 days … Don't leak"* is about another brand). The search
> engine matched the word correctly every time. The problem is that the word isn't the meaning. That is a limit of **any** search built on words, and it is why Workshop 4 learns sentiment from data.

## Recap

| idea | what you saw |
|---|---|
| String ≠ word | "leak" as a string caught *orleaks* and *leakproof*; as an exact word it missed *Leaking* |
| Tokens and types | 1.1 million tokens; 45,310 types with `split()`, 17,602 with a tokeniser that drops punctuation |
| Zipf's law | 100 types make up over half the text; about 42% of types appear once |
| Stop words | about half the tokens, but the list includes *no*, *not* and *n't* |
| Stemming | merges *leaks/leaking/leaked* (recall ↑) but also *general/generic/generous* (precision ↓) |
| Lemmatisation | real words, but needs the part of speech; *was → wa* as a noun |

**After the break** you will use `tokenize` and the Porter stemmer to build an **inverted index** over all 30,000 reviews,
and answer Boolean queries like `leak AND NOT coffee` with it.